In [1]:
import json
with open("intermediate_data/data.json", "r", encoding="utf-8") as f:
    form_data_dict = json.load(f)

with open("intermediate_data/dummy_prompts_data.json", "r", encoding="utf-8") as f:
    prompts_dict = json.load(f)



In [13]:
from typing import Any, Dict


def attach_prompts_to_output(
    output_dict: Dict[str, Any],
    prompts_dict: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Attach prompts to any level of the hierarchy WITHOUT
    wrapping the top-level dictionary.

    - Leaves → {"value": ..., "prompt": ...}
    - Non-leaf nodes may also have "prompt"
    - Root keeps original structure
    """

    def recurse(output_node: Any, prompt_node: Any) -> Any:
        # Case 1: Dict → recurse into children
        if isinstance(output_node, dict):
            merged: Dict[str, Any] = {}

            for key, value in output_node.items():
                child_prompt = (
                    prompt_node.get(key)
                    if isinstance(prompt_node, dict)
                    else None
                )

                merged[key] = recurse(value, child_prompt)

            # If THIS level has a prompt, attach it here
            if isinstance(prompt_node, str):
                merged["prompt"] = prompt_node

            return merged

        # Case 2: Leaf value
        return {
            "value": output_node,
            "prompt": prompt_node if isinstance(prompt_node, str) else None,
        }

    return recurse(output_dict, prompts_dict)

In [ ]:
# form_data_with_prompts_dict= attach_prompts_to_output(form_data_dict, prompts_dict)

In [ ]:
import json
from pathlib import Path

form_data_with_prompts_dict= attach_prompts_to_output(form_data_dict, prompts_dict)

folder = Path("intermediate_data")
folder.mkdir(exist_ok=True)

file_path = folder / "form_data_with_prompts.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(form_data_with_prompts_dict, f, indent=2)
